# Eye-Tracking Autism Screening: Google Colab Setup

Этот ноутбук разворачивает backend/API, пайплайн обработки взгляда и заглушку ML-модели прямо в Google Colab. Запускайте ячейки по порядку (Colab может сбрасывать окружение при простое, в этом случае выполните всё повторно).


In [ ]:
# System packages required for video processing
!sudo apt-get update -y && sudo apt-get install -y ffmpeg


In [ ]:
# Python dependencies (matches pyproject.toml)
%pip install -q fastapi==0.115.0 uvicorn[standard]==0.24.0 sqlmodel==0.0.14 \
    pydantic-settings==2.4.0 passlib[bcrypt]==1.7.4 python-multipart==0.0.9 \
    pyjwt==2.8.0 python-dotenv==1.0.1 opencv-python==4.10.0.84 mediapipe==0.10.14 \
    numpy==1.26.4 scipy==1.11.4 pillow==10.0.1 aiofiles==23.1.0 pytest==7.4.0 httpx==0.26.0


In [ ]:
# Materialize the project tree from an embedded tarball (generated from this repo)
import base64
import io
import tarfile
from pathlib import Path

BUNDLE_B64 = """H4sIAAAAAAAAA+w9a3PbOJL5rF+B49VWUbs0LcmSNaMqTZ3HUWZSk4fPcaZqz6Xi0iQkYUyRHD5ka7O5j/cD7ifeL7luvAhKlBwnjnM7Z36wSaDReHWjH2hA6TrNkt9oULhFsoyefZWnA89xv4//u8NBx/yPT7ffGTzrDrrdwXG/MxwAXPeof9R5Rjpfpzn1p8wLPyPkWXlVxkW5B+6O/H/S51JO/7QV+0tKxsSia3pQZH5wzeL5gV8WLF8e+GlqtVY0y1kSI0zH7bodqxXSPMhYWsjUSRweFMkBjUMiypGM5dcEYCiNARtJI7+YJdmSXPk5DQmUMiuzWhn9vWQZzQ/SdbEQOH8YH7ldXlUKeGkcMJpD+mWLwGPNfJi9lP0whgZ1B9AkR6SXKxYkWXwJuXHoZ+EUIXr9CiD/PVomIY0wHcr2VXq6Dv24YMFBTosC2pT/MO65RrnUz/OIXV1eBdk6LQBt1x26Rmls9sGyjAqW+lkhsH9fZf92UyDC7wyEokiYFDReITpojcpLoMvBSg7GD+M+DERVcElDBl1PKe+82Ye4XKZrRNU7NjocMJHY7RqVsyhKbiAVm6lTfZbMWESx60dinqetlqITN+HT7UcH5oxMYX5W1bRAi2kOPR2aQ7coivSWz4PRrKyczTBtoKopkiRy5fRNW0B3SARXQCBQlwuf7tJn8YjT4yLJC0GMovmtNMkw4TtYU4CUosQP4avISqrwYm3TVsRiehDReF4sIL8LwN+aCb/hI4f28GvWgYv8cDDYtf7jU63/fYDr9rqdo2dk8DUbpZ7/5+u/mn/gqK9GA/eY/+4RvMP8H/eHT/P/GI85/1wi5m66fuA67tD/jga9oZ7/Tgf5vz/oDp70v8d4LMt67hc+KmREzj8ktWZZsiSeNyuLMqOeR9iSC1c/jpPCRwUgb7VkGgV9Q8CHfkELBlqkzFHfIrdYp6gDyry3Uo1oiUylj6nsF4xGoUPOaSRqW7DUIe/+/dVrBGq1WkEEihg5hRwaF++gSWVu50Xm8Na4E/jTHnENA3UUrBb0BPlq8fR5BnoeRQXBkq8iPaQB6gc8Q71busLnzJ/HoHawYF+dYebPuGbCXwRe/8ZnqFB6ZSoVE2sjScCBmhXQPFdN1l8iN0iWaURlu/WHyJv5oLTxDPFWtfp9TjNbDZ5DYLYjOr4AvUi2l4UjPR+XLC6mgIRPgB3SmQ/K7PhNElMH2saWfrb2rulaFOelKWhk0YjASOhiDPTCWw7ikDJmv5fUgF/4+YKGHmrSN0kW8pKiA2UUeWiFGK2BPGwN1i8GIKNAVKHnF6OK3DZa6838oEiy9VgBuGURxMlNu8VR5DigQDgjErG8uLSMORU5FtZoUp6Ni6SXJmkJaTQfWyUMqNVuogqB4esNNtbsIQooqQuCSUXZPOagvG0uCy0Bn3MyHW0R7lalmwAup922pDrOZp5CVmO7LUy1XFcynUCUllmwQNNvRK5AH98q+sKPctlRtBy9HKwAkxpmwCdFjR44WERXNNpDNBkFVllCQ8SMfjXy4hOUhl+IYcVCmgB7FIs9DZULA1SUB36MwPtKaNIx8nFRuJPQFa8YxP4rNu8kKxh24usRuqx5P62HmmgleEX4vmyhB2KH1lcnVb2V+TcSGu1dOYBqNfoyUvjWYv2TH1P/Az6fsfmj63+9wcDw/w07XP/rDJ/0v8d4QNk7SdOIBXwRIIIEykx8gUpWRvSTFUIOMyvjAJ0tuQKJstIL/GAh1UDksohdqdwz+JQZ0vVWUwHrWZ7yyimYH0GYvJNpeoVSCbaZKxcl6MopCKfMj9jfQVnKYE6RrxVe3lWxfqSp1EXE0mFN1pRcSE8lORHuTSk0WS40MBqvWJbEIGmKHStOlAR+ZDlkBfULaeTBmw8r7eTNry/P3755PXlzoUQ3hSWowIVuBzIQpfGcHixpI8J3k9PzyYX3y+SvakUMUGJ4RQL87tHblME8LllcwkKvltjjjtBfpUHglVm0o27Q11lBR4eHh+YCksNi6M8pvrvhlayXOyq9LElgUHC2ARn+s62GkpYSoqgQ517IsrvLHEpoWTajObTwU8tKaNVUFGWeoPp6x3m2sA/kCHBYV0tfTnpsxkAxbJqN12+fT1558Pf9q4kgFqmNakMG2G6kawFK8lAmIeG58GFt5Xg0DhJl2ZTF7OA7SbjQQADKkUthCCjKJ0bBUKHRrE0OfuAKQVUR0Ptr/xoYAOBB5IE9AgRKjIKE3oKWTK4oCl7FL5xNFArE7FaT7C6vobid+hmwQS6NAI7ES64NAa+LGpN977LGZH9K2Vbr3/RqZHfbLRyrOfCYYn+bj5BaMkaSDeWSM64WlnYtx20YbkWLsFrGGvD/olZgMkWW4GLw8G7A+/j/esMuyv/u8Mn/9yhPw/wbWvXD6IJ36H/DYbdfzf8x+v+GnWH/Sf97jAdW8kkcpgkoADmBNZ4s/difo1ypyEC7TO7rGqycfzu1vrpX8BUs17q02oeUeqXY6lWQJ2cvz5FcMweUwOB6DrQbhxd+fp075DnfmYSXF1DcIe+5gMH3Rkw/X1ycTW4Dyk1VR3pMmr2SIHNghZeKrrEpqfKFAxVwgHxZ+vkWnOuXoI9I4NMyQ1GFdrhDpOMIWr5dSOjkqpgpr7ZhAdmKgZ7nGkysSppuKg7lKDeCp6xt7ntotfhCgMqPHmU7zeiM3Y4tc3Ww0Oqf5+NL0xC3pu2WITPr4hUlsEDuQoZtHbIYtKwyKKSTQQlkqA0EudCgTBAhoEMWFCNTwH6o9BlDB7dGcj9aZ064oCZ0RWGiWSwENonYfIHtIzesWEDjyDwCJULuUOuy76gPBL0ALWnBopD8Z78TAClxEobEkOKAbpUBPZqhpcFh0gWoXhj04JMiY2kSEmA3HqFglJtWrxb6rTLust7qyS+UpkZrAuguzcCgYTFPnWX+cqs1J6uEhZxaBLfAOOYYeyEGfwOYa6Q8UqMAzGvCcuJfJStKjjp/Al4PZY8xPY3K+ZzXvd2Pj8aMp0AgMOXap+Sgrp6it9DjjDOWbONu+VPPqR8K4hBeGUWvQi1P/TUy+IjsLH/KS4kWBYLtPOEOM5nQ9DuNDI50Wpzq5O7IFnKlJnKWAorfYjNb4tzQCXmeu9Ehs3UuCx3VOVe7Trd5aPeA4nJ6uXdUp2JY0ROu2pDbu4eoaXikZY3rJkW2gyGwTS0dWMzeNXaVOu/eLICAdwK60vFNxmOyMUYGDuAWgLpa70ZTefVcDF6y221pjNWmRsDSWxrYultt14+ihhVMjdrhh8pl+fGzaFsslYIONtyfzh6y3Tcnn0qUhqwfa9Jsbo/TPP5y7CpEuxm/NlKHcnPhs0YsR0+wRGA3uY2d5gVCblGcifQvWBe+1TCrskJC6CGo8FcLB7p90kKsG36+jgPuIfBSDOYDphfWty3+jQxdyQE4wBRzISw8KXwZxBe5zVnlu8Ls/lT7XbQDxa1W8lwMdrMNlA6xbq6sNsASoCLu86jcFjcLdI4EizK+JqOx2FyVXiMYE+hPt9Prkz8T/Nce1SSbgFUo3ZuMFdTmmNrVRq3CFURJTuvGvNHETRIXHsYddM5VKykgd1C6GPxzmekYG3gwwyEdi3cXFVav1+l5J6enk7OLyXMgRGNqRdOFKreHKfiAGvPNXV0RtV3XbQuISlPwClSsR1ua9n7GQX4Qmrjd3slEdaDHZyg2M5C49Z1O8i9jJfXrm5oyYqAirMxnOa1bErY5c/1OB/mp8Fk0tiQusiy5Y02HIkgfm4wGkP5BNKEA0xIHCubO5u43/KNEV17OQDFXTk5O0/iHR/OCflkpke4y7VttV8JjFk/ZZGWFSXu3DP9cmxySmdoU9Ezy9j7ohn78IKr4KFDn/orv9nNHrGCuzcVntrnctE0qcEED9YOFJGhziatQy8HapFjXD8Wb3WjoOObUI4WAOmwbSCVWteY2cqmp6qjRGNfRagi9J4nYxxuVVWAwjrk/p2OLb7Xi7jVlAOaSsyo2BFRBXBoyoBtXLiqft+nY4P/xwORihec92EbgHf6fTm843PD/HR91j5/8P4/x7PWtbHs6XEEj2ulTFguTi3LQMVLmNfgP2kaGy+IgKkMqP23EIhBn+8BMv6SC/tbD90//NPA/n46HDAK4K/6z0x9U/N89Av4fDLpP538e5cH9f5hvitvrQv4DzxKqXML33Pvf6aPVLtl7eFulkug8rNtVCOwt+84067isrgaFco3RMaKCPHMv3dFRjTqmERXyvU5UxF55T/nXtL1tMWd0DmoS31LeZTJAy9AedvYYCl3v9HxyAnaCMJoVUt4re8s2RozSX9boWTBcYQgqdGBu44nN6Lr3pOb+Qfh23dGDSS6PIEXPjrJZeUK7DYpsBuOgFXVVy2fq3RNejR+hgbjWw0C1ql0Km8VoWaXc8RaNa+2rFDYduKoBdEoFtBH2Oq4RjK3dfDKhrXQ6Q7NEbdbGRtZTMbaRFXY9EQgNKGZhgEstFhO2CS1K5ui93UVlF0jn0lOIkM1U8wqzdhNNDdnIHPEtXlNmm1OnCMMbqoZJkQasRCLA8ZNJw+SRPvDI+zcn7y9+fnv+8j/AoNY08zLmsRzI9yE20Y/yhnAa6EPDwsAHX9JykyHBB8I2S4zNj21XI4/0uWMhMPdQuGdltze3mZVlI81ST1rWH/cx9T8V9/XQEaB3nf/Gwz5V/Cfu/w+gwJP+9xiPef6nLFjECgz6KrmTQQV3308JDJK4oLeFsdsvU3hkgTYqd2p7VUi5XFNpDHKFNtiiezbGW97uPWiBr1qzxbetQVwz/tEhNFgk6mRCi6+u3D0SXtkb8XQYW8ox6gBKERSfo4ACCbcmYcIFFddjdAydHuUlyBwsqfYEcb9JtI2Lgo1R5Psf0uWUB0lK7SrE9SxL0MHFN5r9OBdbvH5EOBzxxe6vjw42nOxkRiBZ7LbnulkSNw97E75U1RbulszWlahdY4ikKrDpD6vrJ5TLYqJF8mgLPEuiCCfZbtdFuXRaQzei9XYp5ahvbe2ktUeE/CtJM3++9Ecw/ECNK9A6DsgLMFbQ1tHn2NdkkSTXHDXfmtjoN+5EKN1mR9e/NTN/xtMUFPvQddzr/G8P7//oH/W6T/F/j/HsCYp+MDq4//n/4w7e//I0/1//2RNQ/w3nfzAcHj/N/2M8TfP/0BtAd+3/bK//x/3ek//3UZ7a/Q8+ix/+9N/d9l+vd7Rx/0cfNICn+X+MB9R9pQjjubg14X7/L3L7S3y1PHfJwjCiN35GwSDI9Abi6dvzd6911v1svC1YbXZJaGmn3bmJqXccW3sil8VVVLJvdsGKiI61xahOC7Zx/zPlYQdVh+WOQ62ncpMBr9/ykoyBgYEbAX+2pmYGGISLJGzIWFA/pFlDhuGmFNFXLW46YpvATMS458K2eNxAmcpwazQfRcKmPavNXIVCeCGh8qhYyNLiI1hQNNiaY7Mt4Wm1RsRKri0MCEZcm1vAeg6e9nQf9THXf3nzAEspXnvycGbgfey/LsoJeBs86f+P8uyZf3EaNivjGDeKvkAtuEv+9wfdav6Pjp91ep3h0/1Pj/OAoH+JZ0dmfkBJkfDTI0Xm8yuQzEs8l/dyBCsBzP9F7KrprNdJvHbIKcgt9JE65Dlr2uJ36zRZnbe+KlkEMlB7j2W6vgSAu1TPOe1qt+jFgsXkJgO0UFB6QfnBHOEuDcmZuHZU+KV5KKW+D4AHTUu7iB+mduTtCNUtAQ3HqwUIhm6qkXDFmzxkbhs4Kn+n3M9c+LlfFJmEcfA2KIri1cMJsTbCmsV25wnAsysQo5MsS4zta/XMrNeiRR+Mij+KSFR6myY5+oz/ZtZjqxFv/01c7YAXJNXwbhzJ9lTpWTzS03t5qSZoKmb6kl+cBSQwnQpVa19PqxmQ6XICVNNGevr5FNTxGzvCKna+3sqqh0A8HnrdgajH5NIySMia/lG1kj3rv4q//WKL8I71v3d8PNyQ/8Pj4dP6/ygPLHA/s/nigN+hRZIMVHk8FFgkGT8NzEmC/M9//bdmNv7B4hnNaAwiQ6/M97IYP/E4MBcROwTDfQzDDSEy9/9OPXrLzz4mWoTIBI/niui3OzGZKpLCYywbXyzOHMIz9CVf9Vu4qnhweThCSzqoPhcHQcsowlXTuFYQD2/umL89kk53c2R2kPyDCzx5z1iDAOSL7W6jugZWG8xxrVI8rmBUa9fQuua1LYaw0McM1OgpsbF5tm6HHKlOOuEDxmlaFuogQ120ninkVZuMm0l2npoYdfrhR90697fcFK3tpprFcSUP94G12HKLxMOidhtvohSXwozllTDtTeFn4DKlqjgcIU5Y7Bgk81o6vBnnH9WRsF0C1+AkGLIG/rIrpAYtqIVmvEH8tlFyG1zNjBTv21NfHQSqBL5CImmNz5nCUTd/lOKxt+ilVWsNv0iSH/MwU7fmxMSwoYI0s/lDaiN75H99mfwCNeAO+T/oGvf/9o+PwP7r9gZP9388ygPL/qvkRor/KgAIZb+cepQZSApEXRKCMi3zb8SKcD+5jx5aLsAq76tO+vyLQoJVT73qX0XAaI1lip7LqgIhOn+CvpxhV7S0fJNkS3khHO9okPATcxjJk/t4x25YXfRQdbp+PG9E+OWkPPHW/FibH1xpwVgTqlKB319NXlx4k79OvFcnb56/Pjn/5R3y/tGRQ7r8z+B7+NMfTFve+cuffm4CPe45pHcMsEffHcOfYX8qA2E8f0VxS8+ja+qJSyLsCBSApZ/hRSl4BQYsLXwRL0roqLhh1RFtkyv5Lf+9DV3qkk3dW04fDG+bkCimoq/boOsdoMoYK5f2bY4SMqIxvjk8aa2T4E32Zb/8MIWSI+6/8CAF8tXldj3eTX4lgiaBqaaBicAOhnedFsAmZpyCZUzcaySwMySwF35AX9N84QpKuODaljCQ8VYMwI6e+IQHLIkrdLMVMMMKiCohDKjWj/E+NYK//sHv0nBVW1ry1Hp1d92mnJQeAi7+eTRZbrc3Q5/xNO+bpHiBbg7hDjBOSgZ+iryKQXCrnsvFzKlIwiutbXn80ahKFnBZ/jalMQ13x1WJeKocLzpB0NNflQK80b5zcZecaNrMeh+jowBdUHgSXPDZiHyoev5ROQJmKZKZahBuSmAfTk/OvLPzt2fei7N3bdQXjzquuM4QXVsesCmO5jJ18yQqRaibznDVXBqnOXHhCjy2RO5BCS1CAI1Tmv6tF5dLvAKW5uOuY0j1GUhPT3OB3IjR5VjshbSgnFC8akHAH0GpQ6nfxWkEkmNhMIO8z7qibeRFwWyCHfit3JDYaW2H8IkD9djSulspL3lEuuQoY9j5Kft2DVZSiixSx4PPFZS5bm0WMdv2pxrj4vHrzjYas8BfxqS7BYCBkiwuab2qbH7l6U4AvQSr4jSJkPQw0eFpp29fvT33fvzpvHf+04/1zkl9HgpXVCP1ZlujbhwPWdLlvwrEyaWijQfrXR0ttHJfrZedqas+algiOiu8W0f8XwOWJvlRx+WQBvm1MXR4vxKiFS+fjLdB2tUR+6u5hwRti3aTv6iqUHj0tkDXGnStQdcCtAZrHrk3p+MQV54apMF9aN3DGrftctX8aGu8Y/3mkNsx74VD1vxl7ZAat39f73H9q5FWtkJkK46NqK9vs9BEwylZh8/y+RKi2fSD1K0S3SXLIVaDWP7jekv/eM8+/++Gc+qzLcA77L9+t2Pu/3XQ/hs8xX89zoPbYnhGAH+WL6/ZefAnqZwx6EKkeG2JuO/nwaw+Bww19KzIQyFYsHZp4ZL6cZMB+BJEhtg4FKbg5/l89UrWZCgqZ6BhLN7T3AObOglYcacJuLNq+Rs2Ur/jtkutVUK940YqKooZmKxmXaG8yF3XxBPBlioST++jbYTtGBLgQ03aWLVqrJH07ZqJG3cIquoVrPreAJNCY0QuBSnYPKHNDUf+isYjRyAgjSsSP9a6xL2gukugRFY9kvONEK3NbmIitG2ZiluyXTU4bW4gA8mPe8oG9fA3d8qCempm8Uj3iiX4qyi1iZGOcX3p0KjJvnZIWMip4S3mbyNldSnMaExyJzsMR1jgMenO1kwpUydEdUhWCVoeWGMKi3srANYGQLcGsDaFv/2/7R1rb1PJ9Xt+xa0/xVkT7LxYGbmVeKxK1QICtnxAyLnYl8TFiS3bWfDSSBCW7kogKm0/tF/abaX+gADJEp75C9f/qOcx7zt+EWMWeme15Hru3DMz5zFzZubMOdU7c3MLoCZVu/A3G8zNQR3LoAJVOwIV/fdmi0o4jcVIDpUn0VmMurg8n2dn2xan4xWqxibohh0eioCGUbjBQxFar8FKWu7IqO1vtWCmJuAaCODjn+R+sdDIVUF3yfzHsL4lFqSZ8w1rNJQ2AnIRqvge1/mwsMdgNgVDRdT7CPbyzBZfvUSTZODSfm6ywrmgdFADa5u6Q9eKRVyYzqK6ibsn6gWqubO4cEafaFng7ELW2KiX3IyaeV8O55blAgI6Dxor/e3ijo2LC61isvyW7O7YSrJWiwmg1o2tUndKolort8u5XTtXNr0kH+zXhoJNX+vfupzuQYKQUtPnUUr31EITdLnJ8wpJnQyoUUqCu3ascF13GgQyWSJvFDCFVCKVm9IuJb7M2VNDCRggpxpTkg+u3YMEi9q9RTfMsMU+1fQ/9TRA/5/YNZCh9z9WtP1/oYD2/ydOLBdS/X8aCWZPdjNnWAhIDtBzjNyKpMn2Yzc5TRNM1v0v4Tx+0heAx7L/5vhfS2n8j+kkL/0nHAFk8PhfKCws6f2fhfwKjv+Ly2n8t6kkGNBP4ckmDP3k26lCg74ZPeJ2o3ULlm23x97zGRALeMRTfidI8PsH7hCuHIa5kJvs9TPLO8dwU0Dh81h87Lf8GXRBzReN1rAK9Jj1uZ66+tvvSVcg0tGFguiEL2C4ZpDanBv0dYRYCgTc9Ozcr7i9kBTVlsRfe9HH3FFKQmI3YE4sZvtb1YOSevKtFE1cka823Yc+hSzXKIm30oebD4zH3b4kie3+2mvLpwjkxmdVVpxjEshqN57I9/vebIx14cEAJ/a7fPcbRnXntlQ+f+FK+asLX583fLmJJtAW0E20ishYLZARJii4iXbgLLN/VVJoO0LDFqFRl06dO3Pm7HndsPPQILq/GVWTFps+Ejtu/y32Md3xFvvSMRfIuAAslWxTMC7lXX/prl3sIOfpiHLZhgBKRf7SMga7R96Mdgyuf5i0u6f2sllWvtlEf7DswU3UgaGhmU684H4jwMcbRCyf6+OwFdtiKdPgsbhJ2/oIu1nbXNZH734E0avZI9BC+mvX9djBr/t6fNfxpEuONZWXxLKeXxwXMJ6EabIzzHjwOoAvjNoQVpEOnHLvOero2PBooYEnCrMMlm+o69eZbBblud9LlmkVU91XC9vFlpIg6EXGy5xOuHn3a/utH0Sf8O4OKNvY/P0EBbf765E52H2yYxbaDZVvwowaVccaskCFbaMW7L/COQ4yufJReQJr/ThY5xM8b0yMZOQpx8Y26fRQR4UxNxFdbyJYmEHp6C7+RY7oI/ngczwcen3wOZroh1RCiflUlmVBye/FBRKro/Pu/Rp9E8V3/8V1kYh9hrx+5rZGPD9cqUctq0VH4N6hHHwzYwQkifDUtBjchZZuZ/rBGJGzmXEHM7ZZJjGaeGj1XtNHDg/Ws/5BPp0dRh6nhg1RDhXn5uaCs8DOoERW1rc/9gbZZ568+78TdgA3jv+3AuYXVpbT+M/TSSb9Jx74SaRh9C/kV7T/x+UTeP5zIqX/dJLQ3yJ0CN1s1mUQmCbkhWt8Amy6gWlXWlG0SYfE9bCDRqPpkfAnnUz5n3jgJ5GG2X+vgLDL8T+/SPK/kE/lfyopGf9pParj7fnJHPflyLwPlMtOyI/f4m6L19UHgYxgUaRP/aTV7u2O9wBwlKBS0v9kO6pstdDEUR4tYq8XLoo4MqeisCXddWBsGXTUJPztK0+VrW6zc5rzPnS8qqOcOhrHP4NOCxso7AtlagKug3z4mKU4NF+36iJilQgTlJ1p3qbDB0JPycLMLANE15Q3KpifQStnvDuAwX2qJYx11cjIHQhYwdZudo0oSPWwtql+0nZDzg2cpDch8NzCcjVpNGueQTsQE8BkQ9xgTG5VypbcUxN+O+sC9IUDam/d+BPwhehWdKdZg+VamWRjwEmcqpo/KG/UNsl5acmGgNbh2h2pUW3Z/s4AZa7acMnW+bYkJRRXcdngCy28s+Lzkg2NV3EiKhPAu5uBTqK5P3eVroc14Td/tW2iEIR6ntyVRDKaVE53AKQVipVvRd1cENbXGiC56xulzG8vLyyvKO7pGzaqqEWRoi8JjCfJqpBuREG6ngxnP2YUMyPikwhw1S+ivBPfTAaxQlric0I8FBtTZCmXlxO3AtTOth37S4eKxjHBDPTEW/zENkU9IrNrFxml2Ro4stedwM76I0UC+aExMOFn/uhThgPbcqR2wErO+areOxg1oJfev1BBmLfqVcYyBvhCXw9mjC8jbpvwtXs3c/Xq1WPGXIlXYDI8UGa2zXBt1raglg1k+GpEDM9xA4ewO46hzPDXjc2SrRvS04G4ESBDovGeD0iftaMJGYPO073oFkME7UJioy92f3f1Ct2KGLIVCbNmK2x1AxvSkNrYtQb8nDmq3EFf30fqqHQCR8PwY4uUHfVcC4HRTlsQTKnLXp+x4qGPKkMTWXVZ+z8iltyklwBD9P/F/JK2/84vYPyHJfL/m+r/Hz5htKhuNcQhTeqjtOhHh/DnLkzS5m+gaV9TtkG8PgUarYgCRmE7L+MU9BUGW/Jc9WQpU4q6aT+SC9w9bWUnx+EXVUVi+jRVJxpp9ZRYhh6wp1cQ0swNHvltcBd5ME5AlcO21L78+p6CheMFgkjAEeqMRAnlqVijBlCoyguU45PNSvgCqqUYoZN7xDNqfOV6tLnWWS99mbWgYJxJF4Y4LpSTuDgYKGoe4DcE4zQtbwzXeq0NcrECVaMHEqsuCio6Eh6sXnitIWnc9OLVsVEkR//1dtQfikCkF7qB1f4ACIeDPjcRyqpNMcnLhFPLBqto8z/PVepEy2ARvoIqmUQVo5OrPpzE8551QDWgpJcLaHpW50ZHYw/R0X4y5zFwa9fWNqHiG90hoqLxfIlNXxLybB+SDyfSABIMxv0ISNfyQtZsl0SA2JEaLU2P9Pk+ZcMKvh2u8WCXbu5+zonmzolH/LPTOPd/Fhfw/K+wkl9I7/9MIzH9la0ADSS1m7UjRnyw0zD/30sFvf+/WKD4X4WV1P/3VBLq//WwEq03yP+02NBWESEoWgKq7ej7Eh3AhKAojRkMYpBnl0kEfKCbLRHA3sCNwGqtTc5/G9qxv+PX2XCzwb4t2UjtjuNwRPqXEE5HqHzXKN8dobzY1INqxJZet53YpZO+O6TnDfTfgC4xj6ETQHr6gnw6dFUePAm3ZTNGUAQ7XIPTZ+3fBah2uYZOAcjMrWnQHrvDZA7rwR9+L6jMmtmliErC+xoRsE57KXxgRIZ6Jn8cW2uFVWEsPx+cQxd4laj2TdQOQgK2uirbtroaNGinmG5dUBgKgYaQmoxKTqvLNayuaiVqdTUnIGnFCWAhEMiytSXIvxV12453Tc0m6PRiEPfwRhE6ZzPcZCDzstuFeZU7iB2kazXT9AxJWZjP58y2oJuVL4HgVnXkeyXr6Im4Bl2vra1nyMJZw/01FF5hy+ZMvXE741MgrYsS6HGFzBbDAJpRqYXkQgU7c3OrBWRtBdE3YX0rtCN/yEpFW2RjtBUjNeC0cFYIDVjbqoc4vmzWOo0WjCgZsWNq8r52OGRa4RWlkZ7KyuacgmxvVzSaZJZwbOqKLjq47Pb/paLL8/+HsvzhNI79F+t/iyfyqf+HqaSNxo1affIhv600lv6/vID6/4nlNP7vVJKgP27/f7A6xqI/+v9H9/9p/L+pJIP+dMm/0558HcPsv5aXXfovoUugdPyfQsKob7CcwqMNWDGgcee5M0EJFKI/B2irQxodPMoNcvFT7DnLl62oGbaUgog59bCLhyP0g28rC5UPfvOlW/WS/FRnTs7MiKboxSe3h7eDSTfEnUvZyJMz6DquU+dNSoCOGdWoXWnVmuzzUmdXOuFvzN9hs3Z2s0qauZG/rZqA3esYVx2+AskoWs25Rq77ANRdtVnP6GJdUjQsE/8Yv4tf9nbigyA+iF/Dn959yHkR78W7vZ3e4yDejV/Bf7u9B/Hb3o742upFJv4HlN7H1/FB7zsAg8X/As9/DXr3ANgePO5AgdeQAT/oz1P4/QLeHULR+/FbeHoevwt6D+GDXfgJ0Oh7bAY8vu096j0UdRuogbpNsy9+X0FjpUz8k6j6HXTqZ2xA7zEVIEMIAyeKbxy8/Aua9hpxwN9iSw6gLc+xrb37mB2/6j3w40N/uxtA6R0oea/3KH4G3+/3dhAUgQCUHgbxqwAQsQtfHAAikAoBdXkXiRG/UFU43YYxEb3mHafAxo2tjt15p/HejksJcfr9HyLFayLLQbwfYFOgE/FTosxT0Z0+Hf8nEA6LPmSqx2+Qpk/gywNigu+h+/eBO94h3EMsA8CeQeY9INEeE51fYdZ+vGegpve498SPCsMVzHEZu+pcdfu43UGBGbN7UJ+fJYzRwsHOT9Sx54SFd9D0V4icQ2Q1IigxL3EH0NqPIoQAX2HHmK8O+AvgkAeAWBAb5IBDRNx9wg2wSY6x9SLeF0+v4f+fhZA8CQjeG8RZ79FwFNVg5GhtVXilaeHmb6pb7/x44THTgxKrQ8+I2vsCSAIBPyKuGGfI4NhHg/ZCtN4AMoBIOewbNYoE6GWgBBrKI4Le4FvkFaqamOYpwYI2efugh3qnG3/XjOZDaELC96iSZ1TlG6oSBAVHLINjg9531E0cGUGee4/pm5cWfBJ47OIhcY9gKXyJ3wwb/uqNSlgvHj9eCTeiVuiwOhHz0IsGMcN5cPCcGPEFN2MIKf+LfSOGoZ67vYC5YDd+iXTkUQRFDl6TwHO3cWQWfSTiIzJfIOlB2nd4AB1P5mlvtg/heRp3uvxvt0akiDH+jirGvQdE6Wc0DxInHtBQByxMvcJBj9lkjydHMdTvKcbYh2n4B0bDDzTRHIzVdWf2QzGBNn0vZ69XxIb3jZ7iBP1Eour6yV/s3lInancm7vDPSWOt//JLdP63lK7/p5KY/vhveT0K6x/kAtCw/b+lgkv/peVC6v9vKsm6JINsUKnX0PpWHLNdgZzTlOOzuwsxqps48Ws2xXGUwUvlSIyss47XCFFJyYA/CxDEGYcwYEFP9fSKbZuPM0xh34zRJFodVXbesATH04iFfN5bjAOlXstwcQzQWQoyjVvpDcY0pSlNaUpTmtKUpjSlKU1pSlOa0pSmNKUpTWlKU5rSlKY0fR7pf1kfOK8A8AAA"""

bundle_bytes = base64.b64decode(BUNDLE_B64)
tarfile.open(fileobj=io.BytesIO(bundle_bytes), mode="r:gz").extractall(path=Path.cwd())
print("✅ Project files restored:", sorted(Path.cwd().glob('*')))


In [ ]:
# Optional sanity check
!pytest -q


In [ ]:
# Launch the FastAPI app (runs in the background). Stop the cell to terminate the server.
import uvicorn
uvicorn.run("backend.app.main:app", host="0.0.0.0", port=8000, reload=False)


После запуска сервера можно открыть новую ячейку и обратиться к API, например:

```python
import requests
requests.get("http://127.0.0.1:8000/health").json()
```

Для внешнего доступа из Colab подключите Ngrok или `google.colab.output.serve_kernel_port`.
